In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Noisy dataset analysis detection


In [ ]:
# cd /content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection/

/content/drive/MyDrive/KYUTECH/Lab research/Research/Anomaly Detection ADCS/Coding/Notebooks_detection


In [ ]:
import numpy as np
BASE_DIR = "./dataset_windows20/"

X_train = np.load(BASE_DIR + "train/X.npy")
y_train = np.load(BASE_DIR + "train/y_bin.npy")

X_val = np.load(BASE_DIR + "val/X.npy")
y_val = np.load(BASE_DIR + "val/y_bin.npy")

X_test = np.load(BASE_DIR + "test/X.npy")
y_test = np.load(BASE_DIR + "test/y_bin.npy")


In [ ]:
X_train = X_train[:, :, 1:]
X_val   = X_val[:, :, 1:]
X_test  = X_test[:, :, 1:]

print(X_train.shape)


(279986, 20, 16)


In [ ]:
X_train = X_train[:, :, 0:-1]
X_val   = X_val[:, :, 0:-1]
X_test  = X_test[:, :, 0:-1]
print(X_train.shape)

(279986, 20, 15)


In [ ]:
## Preprocessing

In [ ]:
X_train_nom = X_train[y_train == 0]   # solo nominal

mean_feat = X_train_nom.mean(axis=(0,1))   # (F,)
std_feat  = X_train_nom.std(axis=(0,1)) + 1e-8


In [ ]:
def scale_windows(X, mean, std):
    return (X - mean[None, None, :]) / std[None, None, :]

X_train_s = scale_windows(X_train, mean_feat, std_feat)
X_val_s   = scale_windows(X_val,   mean_feat, std_feat)
X_test_s  = scale_windows(X_test,  mean_feat, std_feat)


In [ ]:
import pywt
import numpy as np

def compute_dwt_windows(X, wavelet="db4", level=1):
    """
    X: (N_windows, window_size, n_channels) o (N_windows, window_size)
    return:
        A: Approximation coefficients (low-frequency)
        D: Detail coefficients (high-frequency, concatenated)
    """
    if X.ndim == 2:
        X = X[..., np.newaxis]

    N, W, F = X.shape

    A_list = []
    D_list = []

    for i in range(N):
        A_ch = []
        D_ch = []

        for ch in range(F):
            coeffs = pywt.wavedec(X[i, :, ch], wavelet, level=level)

            A = coeffs[0]                  # Approximation
            D = np.concatenate(coeffs[1:]) # All detail levels

            A_ch.append(A)
            D_ch.append(D)

        A_list.append(np.stack(A_ch, axis=1))
        D_list.append(np.stack(D_ch, axis=1))

    return np.array(A_list), np.array(D_list)
# A_windows, D_windows = compute_dwt_windows(X)
# print("A:", A_windows.shape)
# print("D:", D_windows.shape)
def adaptive_threshold_detector_HF(window, thresholds):
    win_max = np.max(window, axis=0)
    return int(np.any(win_max > thresholds))
def adaptive_threshold_detector_LF(window, thresholds):
    win_min = np.min(window, axis=0)
    return int(np.any(win_min < thresholds))
def compute_thresholds(feature_windows, y_bin, k=2.0, mode="high"):
    # feature_windows: (N, coeffs, F)
    nominal = feature_windows[y_bin == 0]

    vals = np.max(nominal, axis=1) if mode == "high" else np.min(nominal, axis=1)

    mu = np.mean(vals, axis=0)
    sigma = np.std(vals, axis=0)

    if mode == "high":
        return mu + k * sigma
    else:
        return mu - k * sigma



In [ ]:
A_train, D_train = compute_dwt_windows(X_train_s)
A_val,   D_val   = compute_dwt_windows(X_val_s)
A_test,  D_test  = compute_dwt_windows(X_test_s)


In [ ]:
print(D_train.shape)
# (N_samples, n_coeffs, n_channels)


(279986, 13, 15)


In [ ]:
from sklearn.decomposition import PCA
import numpy as np

# Flatten
D_train_flat = D_train.reshape(D_train.shape[0], -1)
D_val_flat   = D_val.reshape(D_val.shape[0], -1)
D_test_flat  = D_test.reshape(D_test.shape[0], -1)

# Normalización
mean = np.mean(D_train_flat, axis=0, keepdims=True)
std  = np.std(D_train_flat, axis=0, keepdims=True) + 1e-8

D_train_n = (D_train_flat - mean) / std
D_val_n   = (D_val_flat   - mean) / std
D_test_n  = (D_test_flat  - mean) / std


In [ ]:
# n_components = 20

pca = PCA(n_components=0.95)
pca.fit(D_train_n)

print("Explained variance:", np.sum(pca.explained_variance_ratio_))


Explained variance: 0.9516988959865246


In [ ]:
# from sklearn.decomposition import PCA

# pca = PCA(n_components=0.95)  # conservar 95% varianza
# A_train_pca = pca.fit_transform(A_train_flat)

# A_val_pca  = pca.transform(A_val_flat)
# A_test_pca = pca.transform(A_test_flat)

print("Componentes seleccionados:", pca.n_components_)
n_components = pca.n_components_


Componentes seleccionados: 119


In [ ]:
import tensorflow as tf

input_dim = D_train_n.shape[1]

model = tf.keras.Sequential([

    # Capa PCA fija
    tf.keras.layers.Dense(
        n_components,
        use_bias=False,
        trainable=False,
        input_shape=(input_dim,)
    ),

    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.layers[0].set_weights([pca.components_.T])


In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    D_train_n,
    y_train,
    validation_data=(D_val_n, y_val),
    epochs=30,
    batch_size=64
)


Epoch 1/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 14s 3ms/step - accuracy: 0.7895 - loss: 0.4452 - val_accuracy: 0.8312 - val_loss: 0.4219
Epoch 2/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8584 - loss: 0.3524 - val_accuracy: 0.8420 - val_loss: 0.3956
Epoch 3/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8619 - loss: 0.3459 - val_accuracy: 0.8357 - val_loss: 0.4100
Epoch 4/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8615 - loss: 0.3429 - val_accuracy: 0.8393 - val_loss: 0.3979
Epoch 5/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8644 - loss: 0.3391 - val_accuracy: 0.8347 - val_loss: 0.4111
Epoch 6/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - accuracy: 0.8648 - loss: 0.3334 - val_accuracy: 0.8371 - val_loss: 0.3988
Epoch 7/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 12s 3ms/step - accuracy: 0.8637 - loss: 0.3319 - val_accuracy: 0.8401 - val_loss: 0.3882
Epoch 8/30
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - accuracy: 0.8649 - loss: 0

In [ ]:
y_pred = (model.predict(D_test_n) > 0.2).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred, digits=4))


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
              precision    recall  f1-score   support

           0     0.7989    0.9954    0.8864     30406
           1     0.9936    0.7423    0.8498     29572

    accuracy                         0.8706     59978
   macro avg     0.8962    0.8688    0.8681     59978
weighted avg     0.8949    0.8706    0.8683     59978



Input
 ↓
Normalización
 ↓
PCA (capa lineal fija)
 ↓
MLP entrenable
 ↓
Output


The first layer of the neural architecture was initialized with the PCA projection matrix and kept frozen during training, embedding the variance-preserving linear transformation directly into the model. Subsequent dense layers were trained to learn nonlinear decision boundaries in the reduced latent space.